In [33]:
path = "../../data/ping_test.log"

import json
from pathlib import Path
from typing import Iterator, Dict, Any
import pandas as pd

def read_ping_log(log_path: Path) -> Iterator[Dict[str, Any]]:
    """
    Reads a ping log file with lines like:
    2025-07-27 11:24:48,594 INFO {"address": "1.1.1.1", ...}
    and yields parsed JSON objects.
    """
    with open(log_path, 'r') as f:
        for line in f:
            json_start = line.find('{')
            if json_start == -1:
                continue
            try:
                yield json.loads(line[json_start:])
            except json.JSONDecodeError:
                continue  # skip malformed lines


In [34]:
entries = []
for entry in read_ping_log(Path("../../data/ping_test.log")):
    entries.append(entry)

In [35]:
df = pd.DataFrame(entries[:10])
df.head()

,host,address,unit,average_latency,success_rate,timestamp,metadata
0,local,192.168.0.1,seconds,0.006227,100.0,2025-07-27T11:29:10.458004,"{'host': 'local', 'address': '192.168.0.1', 'n..."
1,local,192.168.0.1,seconds,0.006227,100.0,2025-07-27T11:29:10.458004,"{'host': 'local', 'address': '192.168.0.1', 'n..."
2,google,8.8.8.8,seconds,0.013358,100.0,2025-07-27T11:29:10.530073,"{'host': 'google', 'address': '8.8.8.8', 'num_..."
3,google,8.8.8.8,seconds,0.013358,100.0,2025-07-27T11:29:10.530073,"{'host': 'google', 'address': '8.8.8.8', 'num_..."
4,cloudflare,1.1.1.1,seconds,0.015513,100.0,2025-07-27T11:29:10.553978,"{'host': 'cloudflare', 'address': '1.1.1.1', '..."


In [36]:
import pandas as pd
import json
from pathlib import Path

def read_ping_log_to_df(log_path: Path) -> pd.DataFrame:
    records = []
    with open(log_path, 'r') as f:
        for line in f:
            json_start = line.find('{')
            if json_start == -1:
                continue
            try:
                entry = json.loads(line[json_start:])
                records.append({
                    "timestamp": pd.to_datetime(entry["timestamp"]),
                    "address": entry["address"],
                    "latency": entry["average_latency"]
                })
            except json.JSONDecodeError:
                continue
    
    df = pd.DataFrame(records)
    if df.empty:
        return df
    # Pivot so each address is a column
    df = df.pivot(index="timestamp", columns="address", values="average_latency")
    return df.sort_index()


In [24]:
# read_ping_log_to_df(Path("../../data/ping_test.log"))

import pandas as pd

# Ensure timestamp is datetime
df["timestamp"] = pd.to_datetime(df["timestamp"])

# Pivot so each address becomes its own column
df_pivot = df.pivot(index="timestamp", columns="address", values="average_latency")

# Sort by time
df_pivot = df_pivot.sort_index()

# (Optional) Resample to regular 10‑second intervals
df_pivot = df_pivot.resample("10S").mean()

print(df_pivot)


ValueError: Index contains duplicate entries, cannot reshape

In [27]:
df_agg = df.groupby(["timestamp", "address"], as_index=False)["average_latency"].mean()
df_pivot = df_agg.pivot(index="timestamp", columns="address", values="average_latency").sort_index()


In [31]:
df_pivot

address,1.1.1.1,192.168.0.1,8.8.8.8
timestamp,,,
2025-07-27 11:29:10.458004,NaN,0.006227,NaN
2025-07-27 11:29:10.530073,NaN,NaN,0.013358
2025-07-27 11:29:10.553978,0.015513,NaN,NaN
2025-07-27 11:29:47.379127,NaN,0.004796,NaN
2025-07-27 11:29:47.481378,NaN,NaN,0.014420
